# Experiment 1 — Contrastive steering vector: story → literal-NL

**Question.** Can we causally close the no-think accuracy gap on the story → Rigid Grammar
formalization task by injecting a single direction in activation space — i.e. is
"represent this as a math problem, not a story" a controllable computation inside the model?

**Design.**

1. **Dataset** — matched (story, literal-NL) pairs of the same implication problem, rendered
   deterministically by `storyform.py` / `literalform.py` from the
   [`semantic-drift-autoformalization`](https://github.com/shivam-raval96/semantic-drift-autoformalization)
   repo. Split into a **fit** pile (builds the vector) and a held-out **eval** pile (all accuracy numbers).
2. **Baselines** — on the eval pile, measure the no-think story accuracy (floor), thinking story
   accuracy (ceiling), and no-think literal-NL accuracy (sanity check), graded syntactically by `checkform.py`.
3. **Steering vector** — per layer, mean residual-stream activation of literal-NL texts minus mean
   activation of story texts, over the fit pile.
4. **Intervention** — add `α · v_L` to the residual stream during no-think generation on held-out
   story prompts; sweep layer and α; grade as usual.
5. **Controls** — random direction of matched norm at the best cell.

**Model.** `Qwen/Qwen3-4B` — open weights, native thinking / no-think modes.

**Runtime.** Colab GPU runtime (L4 or A100 recommended; T4 works with smaller batches).
Outputs are checkpointed to Google Drive (if mounted) so every expensive section can resume
after a runtime reset.

**Kill criteria** (checked after the baselines):

- No meaningful gap between no-think floor and thinking ceiling → nothing to close, stop.
- Literal-NL accuracy is not clearly above the story floor → the premise (translation is easy
  once abstraction is done) fails for this model, stop.
- Per-pair difference vectors do not cohere into a consistent direction → the mean vector is
  unlikely to steer anything; treat steering results as exploratory only.

In [ ]:
# ------------------------------ Configuration ------------------------------

REPO_URL = "https://github.com/shivam-raval96/semantic-drift-autoformalization.git"
REPO_COMMIT = "c9e128f247b6daff5b47fd9711ded5e02d1095e9"  # pinned for reproducibility

MODEL_NAME = "Qwen/Qwen3-4B"

SEED = 0

# Dataset: PER_BIN pairs per total-operation bin 1..8 (stratified sampling),
# vacuous pairs dropped, then alternate split into fit / eval piles.
PER_BIN = 20

# Residual-stream read/steer points, as hidden_states indices: hidden_states[L]
# is the output of decoder layer L-1 (1..36 for Qwen3-4B).
LAYERS = [6, 12, 18, 24, 30]

# Steering strengths: multiples of the raw mean-difference vector.
ALPHAS = [1.0, 2.0, 4.0, 8.0, 16.0]

COARSE_N = 20      # eval-subset size for the coarse (layer, alpha) sweep
TOP_CELLS = 3      # sweep cells promoted to the full eval pile

MAX_NEW_TOKENS_NOTHINK = 256
MAX_NEW_TOKENS_THINK = 4096
BATCH_SIZE = 8

# True: leave the prompt's forward pass untouched, steer only decode steps.
STEER_GENERATED_ONLY = False

# Smoke-test mode: tiny dataset and sweep, to validate the plumbing end to
# end in a few minutes before committing to a full run.
QUICK_TEST = False
if QUICK_TEST:
    PER_BIN, LAYERS, ALPHAS = 2, [12, 24], [4.0, 8.0]
    COARSE_N, TOP_CELLS = 6, 1

In [ ]:
# ------------------------- Environment and outputs -------------------------
# Installs dependencies, clones the (public) repo that provides the dataset
# renderers and the grader, and picks an output directory. If Google Drive
# mounts, everything expensive is checkpointed there and survives a runtime
# reset; otherwise outputs stay in the local (ephemeral) filesystem.

%pip install -q -U "transformers>=4.51" accelerate

import json
import subprocess
import sys
from pathlib import Path

if not Path("semantic-drift-autoformalization").exists():
    subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
subprocess.run(
    ["git", "-C", "semantic-drift-autoformalization", "checkout", "-q", REPO_COMMIT],
    check=True,
)
sys.path.insert(0, str(Path("semantic-drift-autoformalization/informalizing-etp").resolve()))

from benchmark import drop_vacuous, load_equations, sample_pairs_stratified, wrap_prompt
from checkform import grade

import torch
from tqdm.auto import tqdm

OUT_DIR = Path("exp01-outputs")
try:
    from google.colab import drive

    drive.mount("/content/drive")
    OUT_DIR = Path("/content/drive/MyDrive/mech-interp-experiments/exp01-steering")
except Exception as error:
    print(f"Google Drive not available ({error}); using local {OUT_DIR}/")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("outputs ->", OUT_DIR.resolve())

In [ ]:
# ------------------------------- Load the model -------------------------------

from transformers import AutoModelForCausalLM, AutoTokenizer

use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
dtype = torch.bfloat16 if use_bf16 else torch.float16

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype, device_map="auto")
model.eval()

NUM_LAYERS = model.config.num_hidden_layers
assert max(LAYERS) <= NUM_LAYERS, f"LAYERS must be <= {NUM_LAYERS}"
print(
    f"{MODEL_NAME}: {NUM_LAYERS} layers, hidden size {model.config.hidden_size}, "
    f"{dtype}, device {model.device}"
)

## Dataset

Each problem is "if a world always obeys rule E, must it also obey rule F?", rendered two ways
by deterministic code (no LLM anywhere, so both versions are guaranteed to mean the same thing):

- **Story**: a themed narrative (paint workshop, tea house, ...) with no visible math.
- **Literal-NL**: dry English that openly talks about "the operation", ordered inputs, and
  variables x, y, z.

Sampling is stratified over problem complexity (total operation count 1–8) with the seeded,
form-independent sampler from `benchmark.py`, so the story and literal-NL draws cover the
*identical* pair set. Pairs containing a vacuous law (`x = x`, `x = y`) are dropped, following
the repo's convention.

The problems are then split alternately (which keeps both piles balanced across complexity bins):

- **fit pile** — only used to compute the steering vector;
- **eval pile** — only used to measure accuracy, so a positive steering result cannot be
  memorization of the fitting problems.

In [ ]:
# ------------------------------ Build the dataset ------------------------------

equations, equations_sha = load_equations()
print(f"ETP equation list: {len(equations)} equations (sha256 {equations_sha[:12]})")

# Same seed => same pair set in both forms; only the rendering differs.
story_samples = drop_vacuous(sample_pairs_stratified(equations, PER_BIN, SEED, form="story"))
literal_samples = drop_vacuous(sample_pairs_stratified(equations, PER_BIN, SEED, form="literal"))
assert [s["pair_id"] for s in story_samples] == [s["pair_id"] for s in literal_samples]

fit_story, eval_story = story_samples[0::2], story_samples[1::2]
fit_literal, eval_literal = literal_samples[0::2], literal_samples[1::2]
print(f"{len(story_samples)} matched pairs -> fit {len(fit_story)}, eval {len(eval_story)}")

example = story_samples[0]
print(f"\n--- example pair {example['pair_id']} "
      f"(E: {example['metadata']['equation_e']}  |  F: {example['metadata']['equation_f']}) ---")
print("\n[story]\n" + example["story"])
print("\n[literal-NL]\n" + literal_samples[0]["story"])

## Generation, grading, and steering harness

- Prompts come from the repo's templates (`formalize_prompt.md` / `literal_prompt.md`), wrapped
  with the benchmark's uniform regime wording; thinking vs no-think is switched natively via the
  Qwen3 chat template (`enable_thinking`), with the vendor-recommended sampling parameters.
- Grading is `checkform.grade`: purely syntactic, `correct` / `wrong` / `unparseable`, with the
  accepted symmetries (variable renaming, side swap, uniform dualization) built in.
- Steering is a forward hook on one decoder layer that adds `α · v` to the residual stream at
  every token position, active for prompt processing and decoding alike (configurable).
- Every condition is written to `OUT_DIR/<name>.jsonl` and reloaded from there on rerun, so an
  interrupted session resumes instead of regenerating.

In [ ]:
# ------------------- Generation, grading, and steering harness -------------------

from contextlib import contextmanager


def build_chat(prompt_text, thinking):
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt_text}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=thinking,
    )


@contextmanager
def steering(layer=None, vector=None, alpha=0.0):
    """Add alpha * vector to the residual stream at hidden_states index `layer`
    (i.e. to the output of decoder layer `layer - 1`) for every forward pass
    inside the context."""
    if layer is None or vector is None or alpha == 0.0:
        yield
        return
    v = vector.to(model.device)

    def hook(module, args, output):
        hidden = output[0] if isinstance(output, tuple) else output
        if STEER_GENERATED_ONLY and hidden.shape[1] > 1:
            return output  # skip the prompt's prefill pass
        hidden = hidden + alpha * v.to(hidden.dtype)
        if isinstance(output, tuple):
            return (hidden,) + tuple(output[1:])
        return hidden

    handle = model.model.layers[layer - 1].register_forward_hook(hook)
    try:
        yield
    finally:
        handle.remove()


@torch.no_grad()
def generate_batch(prompts, thinking, max_new_tokens, steer=None):
    tokenizer.padding_side = "left"
    texts = [build_chat(p, thinking) for p in prompts]
    encoded = tokenizer(texts, return_tensors="pt", padding=True).to(model.device)
    # Vendor-recommended sampling for Qwen3 (greedy decoding is advised against).
    sampling = (
        dict(do_sample=True, temperature=0.6, top_p=0.95, top_k=20)
        if thinking
        else dict(do_sample=True, temperature=0.7, top_p=0.8, top_k=20)
    )
    layer, vector, alpha = steer if steer is not None else (None, None, 0.0)
    with steering(layer, vector, alpha):
        output = model.generate(
            **encoded,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
            **sampling,
        )
    completions = output[:, encoded["input_ids"].shape[1]:]
    return tokenizer.batch_decode(completions, skip_special_tokens=True)


def bucket(row):
    if row["status"] != "correct":
        return row["status"]  # "wrong" or "unparseable"
    transform = row["transform"]
    if transform["dual"]:
        return "correct-dualized"
    if transform["swap_e"] or transform["swap_f"]:
        return "correct-swapped"
    return "exact"


def correct_rate(rows):
    return sum(r["status"] == "correct" for r in rows) / len(rows)


def unparseable_rate(rows):
    return sum(r["status"] == "unparseable" for r in rows) / len(rows)


def run_condition(name, samples, form, thinking, steer=None, max_new_tokens=None):
    """Generate and grade one condition over `samples`, cached as OUT_DIR/<name>.jsonl."""
    path = OUT_DIR / f"{name}.jsonl"
    if path.exists():
        rows = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
        if len(rows) == len(samples):
            print(f"[{name}] {len(rows)} cached rows, correct {correct_rate(rows):.1%}")
            return rows
    if max_new_tokens is None:
        max_new_tokens = MAX_NEW_TOKENS_THINK if thinking else MAX_NEW_TOKENS_NOTHINK
    regime = "on" if thinking else "off"
    # Empty model arg: thinking is switched via the chat template, not /no_think.
    prompts = [wrap_prompt(s["prompt"], regime, "", form) for s in samples]
    torch.manual_seed(SEED)
    rows = []
    with path.open("w") as fh:
        for i in tqdm(range(0, len(prompts), BATCH_SIZE), desc=name, leave=False):
            responses = generate_batch(prompts[i:i + BATCH_SIZE], thinking, max_new_tokens, steer)
            for sample, response in zip(samples[i:i + BATCH_SIZE], responses):
                verdict = grade(response, sample["metadata"])
                row = {
                    "pair_id": sample["pair_id"],
                    "ops_total": sample.get("ops_total"),
                    "status": verdict["status"],
                    "transform": verdict["transform"],
                    "response": response,
                }
                rows.append(row)
                fh.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(f"[{name}] correct {correct_rate(rows):.1%}, "
          f"unparseable {unparseable_rate(rows):.1%} ({len(rows)} pairs)")
    return rows

## Baselines and kill criteria

Three numbers on the eval pile, before touching any internals:

1. **floor** — no-think, story → RG (what we are trying to raise);
2. **ceiling** — thinking, story → RG (the target);
3. **literal sanity** — no-think, literal-NL → RG (the premise: translation is easy once
   abstraction is done).

The thinking condition generates long reasoning traces, so it is the slowest cell in the
notebook (budget ~10–30 min depending on GPU). All three are cached after the first run.

In [ ]:
# ------------------------------- Run the baselines -------------------------------

from collections import Counter

baselines = {
    "story no-think (floor)": run_condition(
        "baseline-story-nothink", eval_story, "story", thinking=False),
    "literal-NL no-think (sanity)": run_condition(
        "baseline-literal-nothink", eval_literal, "literal", thinking=False),
    "story thinking (ceiling)": run_condition(
        "baseline-story-think", eval_story, "story", thinking=True),
}

print(f"\n{'condition':<32} {'correct':>8}  buckets")
for name, rows in baselines.items():
    counts = Counter(bucket(r) for r in rows)
    print(f"{name:<32} {correct_rate(rows):>7.1%}  {dict(counts)}")

floor = correct_rate(baselines["story no-think (floor)"])
ceiling = correct_rate(baselines["story thinking (ceiling)"])
literal_rate = correct_rate(baselines["literal-NL no-think (sanity)"])
gap = ceiling - floor

print(f"\nfloor {floor:.1%}  |  ceiling {ceiling:.1%}  |  gap {gap:+.1%}  |  literal-NL {literal_rate:.1%}")
if gap < 0.05:
    print("KILL CRITERION: no meaningful no-think -> thinking gap on this model; "
          "there is nothing for steering to close.")
if literal_rate < floor + 0.05:
    print("KILL CRITERION: literal-NL is not clearly easier than the story arm for this "
          "model; the premise of the contrast fails.")

## Steering vectors

For every fit-pile problem, one forward pass on the **bare story text** and one on the **bare
literal-NL text** (deliberately *not* the full formalization prompts — the two arms use
different prompt templates, and contrasting full prompts would bake template wording into the
vector rather than the story-vs-abstract difference).

At each probed layer we record two summary positions per text — the final token and the mean
over all tokens — and compute

$$v_L \;=\; \operatorname{mean}(h_L^{\text{literal}}) \;-\; \operatorname{mean}(h_L^{\text{story}})$$

Problem-specific content (which rule, which theme) differs across the fit pile and cancels in
the averages; what survives is what all pairs share — the story-vs-abstract representation
difference. The last-token vector is the primary one; the mean-pool variant is saved alongside.

**Coherence diagnostic** (cheap, before any generation): the mean pairwise cosine similarity of
the per-problem difference vectors. If individual differences point every which way, their mean
is mush and steering will likely do nothing.

In [ ]:
# --------------------- Extract activations, build the vectors ---------------------


@torch.no_grad()
def collect_residuals(texts, layers):
    """Residual-stream summaries per text: last-token and mean-pooled vectors,
    plus last-token norms, at each requested hidden_states index."""
    tokenizer.padding_side = "right"  # keeps last-token indexing by length valid
    last = {L: [] for L in layers}
    pooled = {L: [] for L in layers}
    norms = {L: [] for L in layers}
    for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="activations", leave=False):
        encoded = tokenizer(texts[i:i + BATCH_SIZE], return_tensors="pt", padding=True)
        encoded = encoded.to(model.device)
        output = model(**encoded, output_hidden_states=True)
        mask = encoded["attention_mask"]
        lengths = mask.sum(dim=1)
        row_idx = torch.arange(mask.size(0), device=mask.device)
        for L in layers:
            hidden = output.hidden_states[L].float()
            last_tok = hidden[row_idx, lengths - 1]
            last[L].append(last_tok.cpu())
            mean_tok = (hidden * mask.unsqueeze(-1)).sum(dim=1) / lengths.unsqueeze(-1)
            pooled[L].append(mean_tok.cpu())
            norms[L].append(last_tok.norm(dim=-1).cpu())
        del output
    concat = lambda parts: {L: torch.cat(v) for L, v in parts.items()}
    return concat(last), concat(pooled), concat(norms)


story_last, story_pooled, story_norms = collect_residuals(
    [s["story"] for s in fit_story], LAYERS)
literal_last, literal_pooled, literal_norms = collect_residuals(
    [s["story"] for s in fit_literal], LAYERS)

steer_vec = {L: literal_last[L].mean(0) - story_last[L].mean(0) for L in LAYERS}
steer_vec_pooled = {L: literal_pooled[L].mean(0) - story_pooled[L].mean(0) for L in LAYERS}
torch.save({"last": steer_vec, "pooled": steer_vec_pooled}, OUT_DIR / "steering_vectors.pt")

print(f"{'layer':>5} {'|v|':>9} {'median |h|':>11} {'relative':>9} {'pairwise cos':>13}")
coherence = {}
for L in LAYERS:
    diffs = literal_last[L] - story_last[L]
    unit = diffs / diffs.norm(dim=-1, keepdim=True)
    cos = unit @ unit.T
    n = cos.size(0)
    coherence[L] = ((cos.sum() - n) / (n * (n - 1))).item()
    resid = torch.cat([story_norms[L], literal_norms[L]]).median().item()
    vnorm = steer_vec[L].norm().item()
    print(f"{L:>5} {vnorm:>9.2f} {resid:>11.1f} {vnorm / resid:>9.3f} {coherence[L]:>13.3f}")

if max(coherence.values()) < 0.2:
    print("\nWARNING: per-pair differences are barely coherent at every layer; "
          "the mean vector may not represent a consistent direction.")

## Intervention

First a qualitative look at one held-out story (does steering visibly change the output without
breaking it?), then a **coarse sweep** over every (layer, α) cell on a small eval subset, and a
**full run** of the most promising cells on the whole eval pile.

Controls:

- **Random direction** of matched norm at the best cell — rules out "any perturbation of this
  size helps".
- A shuffled-pairing vector is *not* run, deliberately: the mean of differences is
  pairing-invariant (mean(literal) − mean(story) is the same however the pairs are matched), so
  for a mean-difference vector that control is mathematically identical to the real one.
- The `unparseable` rate is the coherence canary: if a strong α raises "correct" but also
  explodes "unparseable", the vector is breaking the model, not steering it.

In [ ]:
# ----------------------- Qualitative demo on one held-out story -----------------------

demo = eval_story[0]
demo_prompt = wrap_prompt(demo["prompt"], "off", "", "story")
demo_layer = LAYERS[len(LAYERS) // 2]
demo_alpha = ALPHAS[len(ALPHAS) // 2]

torch.manual_seed(SEED)
plain = generate_batch([demo_prompt], thinking=False, max_new_tokens=MAX_NEW_TOKENS_NOTHINK)[0]
torch.manual_seed(SEED)
steered = generate_batch(
    [demo_prompt], thinking=False, max_new_tokens=MAX_NEW_TOKENS_NOTHINK,
    steer=(demo_layer, steer_vec[demo_layer], demo_alpha))[0]

print(f"pair {demo['pair_id']}  (truth: ASSUME {demo['metadata']['canonical_e']}  "
      f"ASK {demo['metadata']['canonical_f']})")
print(f"\n[no steering]  graded: {grade(plain, demo['metadata'])['status']}\n{plain.strip()}")
print(f"\n[layer {demo_layer}, alpha {demo_alpha:g}]  "
      f"graded: {grade(steered, demo['metadata'])['status']}\n{steered.strip()}")

In [ ]:
# ------------------- Coarse sweep: every (layer, alpha) on a subset -------------------

coarse_subset = eval_story[:COARSE_N]
coarse = {}
for L in LAYERS:
    for alpha in ALPHAS:
        rows = run_condition(
            f"coarse-L{L}-a{alpha:g}", coarse_subset, "story", thinking=False,
            steer=(L, steer_vec[L], alpha))
        coarse[(L, alpha)] = correct_rate(rows)

# Baseline rows are in eval-pile order, so the first COARSE_N are the subset.
subset_floor = correct_rate(baselines["story no-think (floor)"][:COARSE_N])
print(f"\ncoarse correct rates ({COARSE_N} eval pairs; unsteered floor on this "
      f"subset: {subset_floor:.0%})")
header = "layer " + "".join(f"{f'a={a:g}':>9}" for a in ALPHAS)
print(header)
for L in LAYERS:
    print(f"{L:>5} " + "".join(f"{coarse[(L, a)]:>9.0%}" for a in ALPHAS))

In [ ]:
# ------------- Full eval pile: top sweep cells + random-direction control -------------

# Best cells first; ties broken toward smaller alpha (gentler intervention).
ranked = sorted(coarse.items(), key=lambda item: (-item[1], item[0][1]))
top_cells = [cell for cell, _ in ranked[:TOP_CELLS]]
print("promoted to full eval:", [f"L{L} a{a:g}" for L, a in top_cells])

full = {}
for L, alpha in top_cells:
    full[(L, alpha)] = run_condition(
        f"steer-L{L}-a{alpha:g}", eval_story, "story", thinking=False,
        steer=(L, steer_vec[L], alpha))

best_layer, best_alpha = max(full, key=lambda cell: correct_rate(full[cell]))
best_rows = full[(best_layer, best_alpha)]
print(f"\nbest cell: layer {best_layer}, alpha {best_alpha:g} "
      f"-> correct {correct_rate(best_rows):.1%}")

# Control: random direction, norm-matched to the best cell's vector.
generator = torch.Generator().manual_seed(SEED)
random_vec = torch.randn(steer_vec[best_layer].shape, generator=generator)
random_vec *= steer_vec[best_layer].norm() / random_vec.norm()
control_rows = run_condition(
    f"control-random-L{best_layer}-a{best_alpha:g}", eval_story, "story", thinking=False,
    steer=(best_layer, random_vec, best_alpha))

## Analysis

- **Dose–response**: correct rate vs α, one line per layer (coarse sweep), against the floor,
  ceiling, and literal-NL reference lines.
- **Verdict breakdown** for the headline conditions — watch whether steering converts `wrong`
  into `correct`, or merely reshuffles into `unparseable`.
- **Headline number**: the fraction of the floor → ceiling gap closed by the best steering cell,
  versus the random-direction control.

In [ ]:
# ------------------------------ Plots and summary ------------------------------

import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

# Dose-response (coarse sweep)
for L in LAYERS:
    ax1.plot(ALPHAS, [coarse[(L, a)] for a in ALPHAS], marker="o", label=f"layer {L}")
ax1.axhline(floor, color="gray", linestyle="--", label=f"floor {floor:.0%}")
ax1.axhline(ceiling, color="black", linestyle="--", label=f"ceiling {ceiling:.0%}")
ax1.axhline(literal_rate, color="green", linestyle=":", label=f"literal-NL {literal_rate:.0%}")
ax1.set_xscale("log", base=2)
ax1.set_xlabel("alpha")
ax1.set_ylabel("correct rate")
ax1.set_title(f"Dose-response, coarse sweep ({COARSE_N} eval pairs)")
ax1.legend(fontsize=8)

# Verdict breakdown for the headline conditions (full eval pile)
conditions = {
    "floor (no steer)": baselines["story no-think (floor)"],
    f"steer L{best_layer} a{best_alpha:g}": best_rows,
    "random control": control_rows,
    "ceiling (thinking)": baselines["story thinking (ceiling)"],
}
bucket_names = ["exact", "correct-swapped", "correct-dualized", "wrong", "unparseable"]
colors = ["#2a9d3f", "#8fd18f", "#c9e8c9", "#e07b7b", "#7b7be0"]
bottoms = [0.0] * len(conditions)
for bucket_name, color in zip(bucket_names, colors):
    fractions = [
        sum(bucket(r) == bucket_name for r in rows) / len(rows)
        for rows in conditions.values()
    ]
    ax2.bar(list(conditions), fractions, bottom=bottoms, label=bucket_name, color=color)
    bottoms = [b + f for b, f in zip(bottoms, fractions)]
ax2.set_ylabel("fraction of eval pile")
ax2.set_title("Verdict breakdown (full eval pile)")
ax2.legend(fontsize=8)
ax2.tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.savefig(OUT_DIR / "exp01-results.png", dpi=150)
plt.show()

# Headline numbers
best_rate = correct_rate(best_rows)
control_rate = correct_rate(control_rows)
gap_closed = (best_rate - floor) / gap if gap > 0 else float("nan")
summary = {
    "model": MODEL_NAME,
    "seed": SEED,
    "eval_pairs": len(eval_story),
    "fit_pairs": len(fit_story),
    "floor": floor,
    "ceiling": ceiling,
    "literal_nl": literal_rate,
    "coarse": {f"L{L}-a{a:g}": rate for (L, a), rate in coarse.items()},
    "best_cell": {"layer": best_layer, "alpha": best_alpha, "correct": best_rate,
                  "unparseable": unparseable_rate(best_rows)},
    "random_control": {"correct": control_rate, "unparseable": unparseable_rate(control_rows)},
    "gap_closed": gap_closed,
    "coherence": coherence,
}
(OUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2))

print(f"floor {floor:.1%} -> best steer {best_rate:.1%} (ceiling {ceiling:.1%})")
print(f"gap closed: {gap_closed:.1%}   |   random control: {control_rate:.1%} "
      f"({control_rate - floor:+.1%} vs floor)")
print(f"summary written to {OUT_DIR / 'summary.json'}")

## Interpretation notes

Fill in after the run:

- **Result.** Floor / ceiling / literal-NL rates; best (layer, α); fraction of the gap closed;
  random-control comparison.
- **Coherence vs effect.** Did the layers with the most coherent per-pair differences steer best?
- **Failure mode.** At strong α, did accuracy fall to `wrong` (semantic drift) or `unparseable`
  (broken generation)?
- **Verdict.** Causal evidence that a single "represent as a math problem" direction is
  controllable, a null result, or inconclusive (e.g. kill criterion fired at the baselines).

Follow-ups if positive: mean-pool vector variant (already saved in `steering_vectors.pt`),
`STEER_GENERATED_ONLY=True` variant, per-complexity-bin breakdown of who gets rescued, and the
SAE version of the same question (experiment 5).